# GROUP BY y Agregaciones

- COUNT, SUM, AVG, MIN, MAX — syntax for SQLite, PostgreSQL y MySQL

## Introducción

- GROUP BY collapses multiple rows into one result per group
- Aggregation functions (COUNT, SUM, AVG, MIN, MAX) calculate values across groups
- HAVING filters groups AFTER aggregation; WHERE filters rows BEFORE aggregation
- Essential for summarizing data, getting totals, averages, and counts by category

### Objetivos de Aprendizaje

- Entender la diferencia entre WHERE y HAVING
- Usar COUNT, SUM, AVG para agregar datos
- Aplicar GROUP BY para agrupar por categoría
- Conocer las diferencias de sintaxis entre SQLite, PostgreSQL y MySQL

### ¿Qué es GROUP BY?

> GROUP BY divide las filas en grupos basados en una o más columnas. Cada grupo produce una fila de resultado. Las funciones de agregación (COUNT, SUM, AVG, etc.) se aplican a cada grupo. Es como "aplanar" múltiples filas en un solo resumen por grupo.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE empleados (
    id INTEGER PRIMARY KEY, nombre TEXT, departamento TEXT, salario REAL)''')
cursor.executemany('INSERT INTO empleados VALUES (?,?,?,?)', [
    (1,'Ana','Ventas',85000),(2,'Luis','Ventas',72000),
    (3,'Sara','Ventas',91000),(4,'Pedro','Marketing',68000),
    (5,'María','Marketing',79000),(6,'Carlos','Ventas',88000),
    (7,'Elena','RRHH',45000),(8,'Diego','RRHH',52000),
])

In [ ]:
# Count employees by department
cursor.execute("""
    SELECT department, COUNT(*) AS employee_count
    FROM employees
    GROUP BY department
    ORDER BY department;
""")
print("Count employees by department:")
print(f"{'Department':<15} {'Count':>10}")
print("-" * 25)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>10}")

In [ ]:
# Multiple aggregations by department
cursor.execute("""
    SELECT 
        department,
        COUNT(*) AS employee_count,
        SUM(salary) AS total_salary,
        AVG(salary) AS average_salary,
        MIN(salary) AS min_salary,
        MAX(salary) AS max_salary
    FROM employees
    GROUP BY department
    ORDER BY department;
""")
print("\nAggregations by department:")
print(f"{'Department':<15} {'Count':>6} {'Total':>10} {'Average':>10} {'Min':>8} {'Max':>8}")
print("-" * 60)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>6} ${row[2]:>8,.0f} ${row[3]:>8,.0f} ${row[4]:>6,.0f} ${row[5]:>6,.0f}")

### WHERE vs HAVING

> WHERE filtra filas ANTES de la agregación (reduce las filas que entran al GROUP BY). HAVING filtra grupos DESPUÉS de la agregación (filtra los resultados ya formados). Piensa en WHERE como "filtro inicial" y HAVING como "filtro final".

In [ ]:
# WHERE filters BEFORE grouping, HAVING filters AFTER
cursor.execute("""
    SELECT 
        department,
        COUNT(*) AS employee_count,
        AVG(salary) AS avg_salary
    FROM employees
    WHERE salary > 50000  -- filter before grouping
    GROUP BY department
    HAVING AVG(salary) > 70000  -- filter after grouping
    ORDER BY department;
""")
print("Departments with avg salary > 70000 (and individual salary > 50000):")
print(f"{'Department':<15} {'Count':>6} {'Avg Salary':>12}")
print("-" * 35)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>6} ${row[2]:>10,.0f}")

## Best Practices for GROUP BY y Agregaciones

### 1. Always include all non-aggregated columns in GROUP BY
- Every column in SELECT that is NOT an aggregation function (COUNT, SUM, AVG, etc.) MUST appear in GROUP BY.
- PostgreSQL allows omitting columns with functional dependencies, but it's best practice to include all.

### 2. Use HAVING to filter after grouping, WHERE to filter before
- WHERE is applied BEFORE GROUP BY (filters individual rows).
- HAVING is applied AFTER GROUP BY (filters already formed groups).

### 3. Favor WHERE over HAVING when possible
- Filtering with WHERE reduces rows before aggregation, improving performance.
- Example: use WHERE active = true instead of HAVING COUNT(*) > 0.

### 4. Use consistent aggregation expressions
- COUNT(*): counts all rows including NULLs.
- COUNT(column): counts only non-null values.
- COUNT(DISTINCT column): counts unique non-null values.
- SUM(col) / AVG(col) ignore NULLs; use COALESCE if you need NULL as 0.

### 5. Execution order vs. writing order
```
1. FROM / JOIN     → data is loaded
2. WHERE           → rows are filtered
3. GROUP BY        → groups are formed
4. HAVING          → groups are filtered
5. SELECT          → aggregations are calculated
6. ORDER BY        → results are sorted
7. LIMIT / OFFSET  → pagination
```

### 6. Avoid heavy functions inside GROUP BY
- Don't put functions like LOWER(name) or DATE_TRUNC('day', date) directly in GROUP BY; pre-process with CTEs or subqueries.

### 7. Aggregations over JOINs: beware of multiplicative effect
- A JOIN can duplicate rows before aggregation. If joining 2 tables with 1:N relationship, COUNT and SUM inflate.
- Solution: aggregate first in a subquery or CTE, then JOIN.

### 8. NULLs in GROUP BY
- NULL values are grouped together as one group.
- Use COALESCE(col, 'N/A') if you want to treat NULLs as visible category.

### 9. Limit results with ORDER BY + LIMIT
- For top-N per group (e.g., customer with most purchases per country), use window functions (RANK() OVER (PARTITION BY ...)).

### 10. Use column aliases for readability
- Name your aggregations: SUM(amount) AS total_amount.

### Diferencias SQLite / PostgreSQL / MySQL

> GROUP BY y las funciones de agregación son estándar SQL y compatibles en todos los motores. Las diferencias están en:
- PostgreSQL: permite referencias a aliases en HAVING y mejores optimizaciones
- MySQL: modo ONLY_FULL_GROUP_BY puede requerir más columnas en GROUP BY
- SQLite: sintaxis estándar, sin extensiones especiales

In [ ]:
# Count vs Count Distinct
cursor.execute("""
    SELECT 
        department,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT name) AS unique_names,
        SUM(salary) AS total_salary,
        ROUND(AVG(salary), 2) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY department;
""")
print("Aggregation functions comparison:")
print(f"{'Department':<15} {'COUNT(*)':>9} {'COUNT(DISTINCT)':>17} {'SUM':>10} {'AVG':>10}")
print("-" * 65)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>9} {row[2]:>17} ${row[3]:>8,.0f} ${row[4]:>8,.0f}")

# SQLite version
cursor.execute("SELECT sqlite_version()")
print(f"\nSQLite version: {cursor.fetchone()[0]}")

## Ejemplos Prácticos

### Top employees by department

Usando una subconsulta para encontrar el empleado con mayor salario por departamento.

In [ ]:
# Top earner by department using subquery
cursor.execute("""
    SELECT e.department, e.nombre, e.salary
    FROM employees e
    WHERE e.salary = (
        SELECT MAX(e2.salary)
        FROM employees e2
        WHERE e2.department = e.department
    )
    ORDER BY e.department;
""")
print("Top earner by department:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} - ${row[2]:,.0f}")

### Department statistics with ROUND

Redondear resultados y formatear para presentación.

In [ ]:
# Round averages and calculate percentages
cursor.execute("""
    SELECT 
        department,
        COUNT(*) AS headcount,
        ROUND(AVG(salary), 2) AS avg_salary,
        ROUND(SUM(salary) * 100.0 / (SELECT SUM(salary) FROM employees), 2) AS pct_total_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC;
""")
print("Department statistics:")
print(f"{'Department':<15} {'Headcount':>10} {'Avg Salary':>12} {'% Total':>10}")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>10} ${row[2]:>10,.2f} {row[3]:>8.2f}%")

## Errores Comunes

### Error: Column not in GROUP BY
¿Por qué ocurre?
- Intentas seleccionar una columna que no está en GROUP BY ni es una función de agregación.

Solución
- Añade la columna al GROUP BY, o envuélvela en una función de agregación.

### Confundir WHERE con HAVING para filtrar grupos
¿Por qué ocurre?
- WHERE filtra filas individuales antes de agrupar. Si滤波as por una condición que requiere ver el resultado de una agregación (como HAVING AVG(salary) > 50000), WHERE no puede hacerlo.

Solución
- Usa HAVING para filtrar por resultados de funciones de agregación.

### JOINs multiplican filas antes de COUNT
¿Por qué ocurre?
- Si tienes una relación 1:N y haces JOIN antes de agregar, cada fila de la tabla 1 se duplica N veces, inflando el COUNT.

Solución
- Agrega primero en un CTE o subquery, luego haz el JOIN.